# Representational compression of narrative details

This notebook answers one central question: **as a story is compressed into an age-steered summary, are narratively central details preserved or reweighted differently as a function of source age and target condition?**

The analysis deliberately separates four quantities:

1. **Whole-text displacement** — distance from the original-story representation to its summary representation.
2. **Grounding** — for every summary detail, its nearest original detail. This guarantees that every generated detail is accounted for.
3. **Coverage** — for every original detail, similarity to its nearest summary detail. This measures degree of preservation without imposing an arbitrary threshold.
4. **Relative weight change** — change in similarity between an original detail and the whole-text representation, from original story to summary.

Nearest-neighbor matching is the primary threshold-free analysis. Unique Hungarian matching is retained as a robustness check. Mean-pooled representations are a control for final-token representations.

In [ ]:
from pathlib import Path
import os

CACHE_ROOT = Path(os.environ.get('SLURM_TMPDIR', '/tmp')) / f'story-features-{os.getuid()}'
MPL_CACHE = CACHE_ROOT / 'matplotlib'
MPL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(MPL_CACHE)
os.environ.setdefault('MPLBACKEND', 'Agg')

ROOT = Path(os.environ.get('STORY_FEATURES_ROOT', Path.cwd())).resolve()
if not (ROOT / 'mechanistic_interpretability').exists():
    ROOT = ROOT.parent
MODEL = os.environ.get('REPRESENTATION_MODEL', 'gpt_oss_20b')
raw_defaults = {
    'gpt_oss_20b': ROOT / 'mechanistic_interpretability/outputs/02_representation_extraction/gpt_oss_20b_global_current',
    'qwen3_8b': ROOT / 'mechanistic_interpretability/outputs/02_representation_extraction/qwen3_8b_aug27_global',
}
RAW_DIR = Path(os.environ.get('REPRESENTATION_RAW_DIR', raw_defaults[MODEL]))
ANALYSIS_DIR = Path(os.environ.get(
    'REPRESENTATION_ANALYSIS_DIR',
    ROOT / 'mechanistic_interpretability/outputs/02_representation_extraction/analysis' / MODEL,
))

import numpy as np
import pandas as pd
from IPython.display import display

PLOTTING_AVAILABLE = True
STATSMODELS_AVAILABLE = True
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style='whitegrid', context='notebook')
except Exception as error:
    PLOTTING_AVAILABLE = False
    print('Plotting unavailable:', type(error).__name__, error)
try:
    import statsmodels.formula.api as smf
except Exception as error:
    STATSMODELS_AVAILABLE = False
    print('Statsmodels unavailable:', type(error).__name__, error)

def clustered_age_condition_regression(data, outcome, controls=()):
    from scipy.stats import t as student_t
    columns = [outcome, 'condition', 'source_age', 'story_key', *controls]
    frame = data.dropna(subset=columns).copy()
    frame['source_age_centered'] = frame['source_age'] - frame['source_age'].mean()
    design = pd.DataFrame({'Intercept': 1.0}, index=frame.index)
    for condition in ['older', 'younger']:
        indicator = frame['condition'].eq(condition).astype(float)
        design[f'condition_{condition}'] = indicator
        design[f'{condition}_x_source_age'] = indicator * frame['source_age_centered']
    design['source_age_centered'] = frame['source_age_centered']
    for control in controls:
        scale = frame[control].std()
        design[f'z_{control}'] = (frame[control] - frame[control].mean()) / (scale if scale else 1.0)
    x = design.to_numpy(float)
    y = frame[outcome].to_numpy(float)
    beta = np.linalg.pinv(x.T @ x) @ x.T @ y
    residual = y - x @ beta
    bread = np.linalg.pinv(x.T @ x)
    meat = np.zeros((x.shape[1], x.shape[1]))
    groups = frame['story_key'].astype(str).to_numpy()
    unique_groups = np.unique(groups)
    for group in unique_groups:
        use = groups == group
        score = x[use].T @ residual[use]
        meat += np.outer(score, score)
    n, k, g = len(frame), x.shape[1], len(unique_groups)
    correction = (g / (g - 1)) * ((n - 1) / (n - k)) if g > 1 and n > k else 1.0
    covariance = correction * bread @ meat @ bread
    standard_error = np.sqrt(np.maximum(np.diag(covariance), 0))
    statistic = beta / np.maximum(standard_error, 1e-12)
    p_value = 2 * student_t.sf(np.abs(statistic), df=max(g - 1, 1))
    return pd.DataFrame({
        'term': design.columns, 'estimate': beta, 'clustered_se': standard_error,
        't': statistic, 'p_value': p_value, 'clusters': g, 'observations': n,
    })

print('repository:', ROOT)
print('model:', MODEL)
print('raw representations:', RAW_DIR)
print('analysis:', ANALYSIS_DIR)

## Immediate preview from raw metadata

This section works before the representation reducer finishes. It audits usable stories, steering-condition balance, age coverage, token compression, and detail compression. These are behavioral/QC results, not yet tests of hidden-state geometry.

In [ ]:
global_files = sorted(RAW_DIR.glob('global_metadata_*.csv'))
detail_files = sorted(RAW_DIR.glob('contextual_detail_metadata_*.csv'))
if not global_files or not detail_files:
    raise FileNotFoundError(f'Raw metadata not found in {RAW_DIR}')
global_meta = pd.concat([pd.read_csv(path, low_memory=False) for path in global_files], ignore_index=True)
detail_meta = pd.concat([pd.read_csv(path, low_memory=False) for path in detail_files], ignore_index=True)
for frame in [global_meta, detail_meta]:
    frame['source_age'] = pd.to_numeric(frame.get('source_age'), errors='coerce')
    frame['target_age'] = pd.to_numeric(frame.get('target_age'), errors='coerce')
original_meta = global_meta[global_meta['record_type'].eq('original')].drop_duplicates('story_key')
summary_meta = global_meta[global_meta['record_type'].eq('summary')].copy()
record_detail_counts = detail_meta.groupby('record_id').size().rename('n_details')
paired_meta = summary_meta.merge(
    original_meta[['story_key', 'record_id', 'token_count']].rename(
        columns={'record_id': 'original_record_id', 'token_count': 'original_token_count'}),
    on='story_key', how='inner', validate='many_to_one')
paired_meta = paired_meta.rename(columns={'token_count': 'summary_token_count'})
paired_meta = paired_meta.join(record_detail_counts, on='record_id').rename(columns={'n_details': 'n_summary_details'})
paired_meta = paired_meta.join(record_detail_counts, on='original_record_id').rename(columns={'n_details': 'n_original_details'})
paired_meta['token_ratio'] = paired_meta['summary_token_count'] / paired_meta['original_token_count']
paired_meta['detail_ratio'] = paired_meta['n_summary_details'] / paired_meta['n_original_details']
paired_meta['age_shift'] = paired_meta['target_age'] - paired_meta['source_age']
print('global records:', len(global_meta), 'stories:', original_meta['story_key'].nunique(), 'summaries:', len(summary_meta))
print('contextual details:', len(detail_meta))
display(paired_meta.groupby('condition').agg(
    stories=('story_key', 'nunique'), summaries=('record_id', 'nunique'),
    median_token_ratio=('token_ratio', 'median'), median_detail_ratio=('detail_ratio', 'median')).reset_index())

In [ ]:
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    sns.countplot(data=paired_meta, x='condition', order=['younger', 'actual', 'older'], ax=axes[0, 0])
    axes[0, 0].set_title('Available summaries by condition')
    sns.histplot(data=paired_meta, x='source_age', hue='condition', multiple='layer', discrete=True, ax=axes[0, 1])
    axes[0, 1].set_title('Source-age support')
    sns.boxplot(data=paired_meta, x='condition', y='token_ratio', order=['younger', 'actual', 'older'], ax=axes[1, 0])
    axes[1, 0].set_title('Token compression ratio')
    sns.boxplot(data=paired_meta, x='condition', y='detail_ratio', order=['younger', 'actual', 'older'], ax=axes[1, 1])
    axes[1, 1].set_title('Detail-count compression ratio')
    plt.tight_layout()
else:
    display(paired_meta.groupby('condition')[['token_ratio', 'detail_ratio']].describe())

support = (paired_meta.assign(available=1)
           .pivot_table(index='story_key', columns='condition', values='available', aggfunc='max', fill_value=0))
print('paired actual + older stories:', int(((support.get('actual', 0) == 1) & (support.get('older', 0) == 1)).sum()))
print('paired actual + younger stories:', int(((support.get('actual', 0) == 1) & (support.get('younger', 0) == 1)).sum()))

### Statistical QC available immediately

The cluster-robust regressions below test whether output length and number of details differ by steering condition across source age. Because summary length was prompted to approximately 30%, these are manipulation checks rather than the paper's central representational tests.

In [ ]:
metadata_models = {}
if STATSMODELS_AVAILABLE:
    for outcome in ['token_ratio', 'detail_ratio']:
        frame = paired_meta.dropna(subset=[outcome, 'source_age', 'condition']).copy()
        fit = smf.ols(f'{outcome} ~ C(condition) * source_age', data=frame).fit(
            cov_type='cluster', cov_kwds={'groups': frame['story_key']})
        metadata_models[outcome] = fit
        print('\n', outcome)
        display(fit.summary2().tables[1])
else:
    print('Using the NumPy/SciPy story-clustered OLS fallback.')
    for outcome in ['token_ratio', 'detail_ratio']:
        metadata_models[outcome] = clustered_age_condition_regression(paired_meta, outcome)
        print('\n', outcome)
        display(metadata_models[outcome])

## Load compact HPC results

The raw hidden states remain on HPC. The notebook only needs four compact tables produced by the analysis commands in the final section.

In [ ]:
paths = {
    'whole': ANALYSIS_DIR / 'story_summary_global_geometry.csv',
    'story': ANALYSIS_DIR / 'story_representation_compression_metrics.csv',
    'source_details': ANALYSIS_DIR / 'original_detail_representation_coverage.csv.gz',
    'summary_details': ANALYSIS_DIR / 'summary_detail_representation_alignment.csv.gz',
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Run the HPC reduction commands first. Missing:\n' + '\n'.join(missing))
whole = pd.read_csv(paths['whole'], low_memory=False)
story = pd.read_csv(paths['story'], low_memory=False)
source_details = pd.read_csv(paths['source_details'], low_memory=False)
summary_details = pd.read_csv(paths['summary_details'], low_memory=False)
for frame in [whole, story, source_details, summary_details]:
    for column in ['source_age', 'target_age']:
        if column in frame:
            frame[column] = pd.to_numeric(frame[column], errors='coerce')
    if {'source_age', 'target_age'} <= set(frame):
        frame['age_shift'] = frame['target_age'] - frame['source_age']
print({name: len(frame) for name, frame in {
    'whole': whole, 'story': story, 'source_details': source_details, 'summary_details': summary_details
}.items()})

## Cohort audit

Before hypothesis testing, verify the number of stories, summaries, conditions, ages, and represented layers. Younger summaries are expected to be less frequent, so uncertainty—not raw means alone—must guide interpretation.

In [ ]:
audit = (story.groupby('condition', dropna=False)
         .agg(stories=('story_key', 'nunique'), summaries=('record_id', 'nunique'),
              source_age_min=('source_age', 'min'), source_age_max=('source_age', 'max'),
              target_age_min=('target_age', 'min'), target_age_max=('target_age', 'max'))
         .reset_index())
display(audit)
display(pd.crosstab(story['source_age'], story['condition']))

## Q1 — Does the whole representation move differently under age steering?

`eos_cosine` measures similarity between the one-pass original and summary representations. `mean_control_cosine` is the same-model mean-pooling control. Layerwise divergence indicates where the steering-conditioned compression geometry emerges.

In [ ]:
whole_profile = (whole.groupby(['layer', 'condition'], as_index=False)
                 .agg(stories=('story_key', 'nunique'), eos_cosine=('eos_cosine', 'mean'),
                      mean_control_cosine=('mean_control_cosine', 'mean')))
display(whole_profile)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=whole_profile, x='layer', y='eos_cosine', hue='condition', marker='o', ax=axes[0])
sns.lineplot(data=whole_profile, x='layer', y='mean_control_cosine', hue='condition', marker='o', ax=axes[1])
axes[0].set_title('Final-token whole-text similarity')
axes[1].set_title('Mean-pooling control')
plt.tight_layout()

## Q2 — Are central original details preferentially preserved?

Let $O \in \mathbb{R}^{n \times d}$ contain normalized original-detail representations and $S \in \mathbb{R}^{m \times d}$ contain normalized summary-detail representations. The cross-text similarity matrix is $C = SO^\top$.

- **Summary grounding:** each summary detail $i$ selects $\arg\max_j C_{ij}$. Every summary detail is grounded, multiple summary details may select the same source, and there is no threshold.
- **Original coverage:** each original detail $j$ receives $\max_i C_{ij}$. This continuous score measures its best available summary correspondence without declaring it retained or omitted.
- **NN-selected / not NN-selected:** an original detail is operationally selected when at least one summary detail chooses it as its nearest source. This is the threshold-free binary analysis called `selected_by_any_summary_detail`; it should not be interpreted as proof of semantic entailment.
- **Unique robustness check:** Hungarian matching maximizes total cosine subject to one-to-one assignments. It also uses no cosine threshold.

In [ ]:
centrality_bins = pd.qcut(source_details['original_centrality_percentile'], 10, duplicates='drop')
coverage_profile = (source_details.assign(centrality_decile=centrality_bins)
                    .groupby(['condition', 'centrality_decile'], observed=True, as_index=False)
                    .agg(mean_coverage=('coverage_cosine', 'mean'),
                         selected_rate=('selected_by_any_summary_detail', 'mean'),
                         details=('original_detail_index', 'size')))
retention_details = source_details.assign(
    retention_status=np.where(source_details['selected_by_any_summary_detail'], 'NN-selected', 'not NN-selected'))
retention_story = (retention_details.groupby(
    ['story_key', 'record_id', 'condition', 'source_age', 'target_age', 'retention_status'], as_index=False)
    .agg(mean_original_centrality=('original_centrality', 'mean'), details=('original_detail_index', 'size')))
retention_wide = retention_story.pivot_table(
    index=['story_key', 'record_id', 'condition', 'source_age', 'target_age'],
    columns='retention_status', values='mean_original_centrality').reset_index()
if {'NN-selected', 'not NN-selected'} <= set(retention_wide.columns):
    retention_wide['retained_minus_not_retained_centrality'] = (
        retention_wide['NN-selected'] - retention_wide['not NN-selected'])
display(retention_story.groupby(['condition', 'retention_status'])
        .agg(stories=('story_key', 'nunique'), mean_centrality=('mean_original_centrality', 'mean')).reset_index())
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(1, 3, figsize=(21, 5))
    sns.lineplot(data=coverage_profile, x='centrality_decile', y='mean_coverage', hue='condition', marker='o', ax=axes[0])
    sns.lineplot(data=coverage_profile, x='centrality_decile', y='selected_rate', hue='condition', marker='o', ax=axes[1])
    sns.regplot(data=retention_wide, x='source_age', y='retained_minus_not_retained_centrality',
                scatter_kws={'alpha': .25}, ax=axes[2])
    axes[0].set_title('Continuous semantic coverage')
    axes[1].set_title('Threshold-free NN-selection rate')
    axes[2].set_title('Centrality: selected minus not selected')
    for ax in axes[:2]:
        ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()

### Worked nearest-neighbor mapping

Each point below is one summary detail. Its horizontal position is the original detail selected by nearest-neighbor cosine; its vertical position is the summary detail's narrative order. Color is match cosine. A monotonic diagonal indicates preserved chronology, while repeated horizontal positions indicate fusion or repeated grounding onto one source detail.

In [ ]:
example_counts = summary_details.groupby('record_id').size().sort_values()
example_record = example_counts.index[len(example_counts) // 2]
example_mapping = summary_details[summary_details['record_id'].eq(example_record)].sort_values('summary_detail_index')
display(example_mapping[[
    'summary_detail_index', 'summary_detail_text', 'grounded_original_detail_index',
    'grounded_original_detail_text', 'grounding_cosine', 'unique_original_detail_index', 'unique_match_cosine',
]])
if PLOTTING_AVAILABLE:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=example_mapping, x='grounded_original_detail_index', y='summary_detail_index',
                    hue='grounding_cosine', size='grounding_cosine', palette='viridis', sizes=(50, 220))
    plt.title(f'Summary → original nearest-neighbor mapping: {example_record}')
    plt.xlabel('Selected original-detail index')
    plt.ylabel('Summary-detail index')
    plt.tight_layout()

## Q3 — Does a detail gain or lose weight in the compressed whole?

For every original detail, relative weight change is

`cos(original detail, whole summary) − cos(original detail, whole original)`.

Positive values indicate that the detail becomes more aligned with the compressed representation; negative values indicate representational downweighting.

In [ ]:
weight_profile = (source_details.groupby(['story_key', 'record_id', 'condition', 'source_age', 'target_age'], as_index=False)
                  .agg(mean_weight_change=('original_detail_global_weight_change', 'mean'),
                       centrality_weight_correlation=('original_centrality', lambda s: np.nan)))
correlations = (source_details.groupby(['story_key', 'record_id'], sort=False)
                .apply(lambda d: d['original_centrality'].corr(d['original_detail_global_weight_change']))
                .rename('centrality_weight_correlation').reset_index())
weight_profile = weight_profile.drop(columns='centrality_weight_correlation').merge(correlations, on=['story_key', 'record_id'])
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.regplot(data=weight_profile, x='source_age', y='mean_weight_change', scatter_kws={'alpha': .25}, ax=axes[0])
sns.boxplot(data=weight_profile, x='condition', y='centrality_weight_correlation', ax=axes[1])
axes[0].set_title('Average detail reweighting by source age')
axes[1].set_title('Do central details gain more representational weight?')
plt.tight_layout()

## Q4 — Does the age effect change across development?

These plots keep source age and steering condition visible. The inferential model uses story-clustered standard errors so multiple summaries and details from one story are not treated as independent participants. A nonlinear spline can be added after inspecting support across ages.

In [ ]:
outcomes = [
    'mean_coverage_cosine', 'proportion_original_details_selected',
    'centrality_coverage_correlation', 'mean_original_detail_global_weight_change',
]
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    for ax, outcome in zip(axes.flat, outcomes):
        sns.regplot(data=story, x='source_age', y=outcome, scatter=False, ax=ax, color='black')
        sns.scatterplot(data=story, x='source_age', y=outcome, hue='condition', alpha=.25, s=25, ax=ax)
        ax.set_title(outcome.replace('_', ' ').title())
    plt.tight_layout()

models = {}
for outcome in outcomes:
    print('\n', outcome)
    if STATSMODELS_AVAILABLE:
        frame = story.dropna(subset=[outcome, 'source_age', 'condition']).copy()
        model = smf.ols(f'{outcome} ~ C(condition) * source_age + n_original_details + n_summary_details', data=frame)
        models[outcome] = model.fit(cov_type='cluster', cov_kwds={'groups': frame['story_key']})
        display(models[outcome].summary2().tables[1])
    else:
        models[outcome] = clustered_age_condition_regression(
            story, outcome, controls=('n_original_details', 'n_summary_details'))
        display(models[outcome])

### Paired within-story condition tests

These tests compare actual-age summaries with older and younger summaries from the same source story. The story—not the detail—is the sampling unit. Wilcoxon signed-rank tests avoid assuming normally distributed differences, and Benjamini–Hochberg correction controls false discovery rate across outcomes and contrasts.

In [ ]:
from scipy.stats import wilcoxon

paired_rows = []
story_condition = story.groupby(['story_key', 'condition'], as_index=False)[outcomes].mean()
for outcome in outcomes:
    wide = story_condition.pivot(index='story_key', columns='condition', values=outcome)
    for comparison in ['older', 'younger']:
        if not {'actual', comparison} <= set(wide.columns):
            continue
        pair = wide[['actual', comparison]].dropna()
        differences = pair[comparison] - pair['actual']
        statistic, p_value = wilcoxon(differences) if len(pair) and np.any(differences != 0) else (np.nan, np.nan)
        paired_rows.append({
            'outcome': outcome, 'contrast': f'{comparison} - actual', 'n_stories': len(pair),
            'mean_difference': differences.mean(), 'median_difference': differences.median(),
            'wilcoxon_statistic': statistic, 'p_value': p_value,
        })
paired_tests = pd.DataFrame(paired_rows)
usable = paired_tests['p_value'].notna()
p_values = paired_tests.loc[usable, 'p_value'].to_numpy(float)
order = np.argsort(p_values)
ranked = p_values[order] * len(p_values) / np.arange(1, len(p_values) + 1)
ranked = np.minimum.accumulate(ranked[::-1])[::-1]
adjusted = np.empty_like(ranked)
adjusted[order] = np.clip(ranked, 0, 1)
paired_tests.loc[usable, 'p_fdr_bh'] = adjusted
display(paired_tests.sort_values(['outcome', 'contrast']))

## Q5 — Are results robust to matching and representation choices?

Three prespecified checks prevent the result from depending on one arbitrary choice:

- Compare nearest-neighbor `grounding_cosine` with `unique_match_cosine`.
- Repeat the HPC reducer using `--representation mean` as a pooling control.
- Repeat the complete notebook for Qwen3-8B and compare effect direction on the shared story set.

In [ ]:
robustness = (summary_details.groupby(['story_key', 'record_id', 'condition'], as_index=False)
              .agg(nearest_neighbor=('grounding_cosine', 'mean'), unique_matching=('unique_match_cosine', 'mean')))
display(robustness.groupby('condition')[['nearest_neighbor', 'unique_matching']].agg(['count', 'mean', 'std']))
sns.scatterplot(data=robustness, x='nearest_neighbor', y='unique_matching', hue='condition', alpha=.4)
limits = [robustness[['nearest_neighbor', 'unique_matching']].min().min(),
          robustness[['nearest_neighbor', 'unique_matching']].max().max()]
plt.plot(limits, limits, '--', color='black')
plt.title('Nearest-neighbor versus unique matching')
plt.show()

### Cross-model representational agreement

The same original and GPT-generated summary texts are encoded by GPT-OSS and Qwen. Agreement means the two models assign similar compression geometry to the same story-summary pair; disagreement is evidence that stable model behavior/text does not uniquely determine internal representation. This is the direct analysis motivated by *Not all solutions are created equal*, without claiming that these two pretrained models occupy the paper's analytically defined solution manifold.

In [ ]:
model_paths = {
    'gpt_oss_20b': ROOT / 'mechanistic_interpretability/outputs/02_representation_extraction/analysis/gpt_oss_20b/story_representation_compression_metrics.csv',
    'qwen3_8b': ROOT / 'mechanistic_interpretability/outputs/02_representation_extraction/analysis/qwen3_8b/story_representation_compression_metrics.csv',
}
if all(path.exists() for path in model_paths.values()):
    gpt_metrics = pd.read_csv(model_paths['gpt_oss_20b'], low_memory=False)
    qwen_metrics = pd.read_csv(model_paths['qwen3_8b'], low_memory=False)
    comparison_outcomes = [
        'mean_coverage_cosine', 'proportion_original_details_selected',
        'centrality_coverage_correlation', 'mean_original_detail_global_weight_change',
    ]
    shared = gpt_metrics.merge(qwen_metrics, on=['record_id', 'story_key', 'condition'], suffixes=('_gpt', '_qwen'))
    agreement_rows = []
    for outcome in comparison_outcomes:
        agreement_rows.append({
            'outcome': outcome, 'shared_summaries': len(shared),
            'pearson': shared[f'{outcome}_gpt'].corr(shared[f'{outcome}_qwen'], method='pearson'),
            'spearman': shared[f'{outcome}_gpt'].corr(shared[f'{outcome}_qwen'], method='spearman'),
        })
    display(pd.DataFrame(agreement_rows))
    if PLOTTING_AVAILABLE:
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        for ax, outcome in zip(axes.flat, comparison_outcomes):
            sns.scatterplot(data=shared, x=f'{outcome}_gpt', y=f'{outcome}_qwen',
                            hue='condition', alpha=.3, s=20, ax=ax)
            ax.set_title(outcome.replace('_', ' ').title())
        plt.tight_layout()
else:
    print('Cross-model comparison will appear after both reducer jobs finish.')

## Q6 — Does narrative context change how a detail is represented?

For GPT-OSS, the isolated-detail control encodes the same wording without its story. `contextualization_shift = 1 − cosine(contextual, isolated)` measures how strongly narrative context changes that detail. This is not deletion influence: it directly compares the same detail with and without surrounding context.

In [ ]:
context_data = source_details.dropna(subset=['contextualization_shift']).copy()
if context_data.empty:
    print('No isolated-detail controls were supplied to the HPC reducer.')
else:
    context_story = (context_data.groupby(['story_key', 'record_id', 'condition', 'source_age'], as_index=False)
                     .agg(mean_contextualization_shift=('contextualization_shift', 'mean'),
                          centrality_context_correlation=('original_centrality',
                              lambda s: np.nan)))
    correlations = (context_data.groupby(['story_key', 'record_id'], sort=False)
                    .apply(lambda d: d['original_centrality'].corr(d['contextualization_shift']))
                    .rename('centrality_context_correlation').reset_index())
    context_story = context_story.drop(columns='centrality_context_correlation').merge(correlations, on=['story_key', 'record_id'])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.regplot(data=context_story, x='source_age', y='mean_contextualization_shift', ax=axes[0])
    sns.boxplot(data=context_story, x='condition', y='centrality_context_correlation', ax=axes[1])
    axes[0].set_title('Narrative contextualization by source age')
    axes[1].set_title('Does context reshape central details differently?')
    plt.tight_layout()

## Interpretation checklist

A defensible positive result requires convergence across levels:

1. Whole-story displacement differs by age condition.
2. Original centrality predicts continuous coverage or threshold-free selection.
3. Central details show systematically different whole-summary weight changes.
4. The condition × source-age interaction indicates whether the steering effect changes developmentally.
5. Effect direction survives mean pooling, unique matching, and Qwen replication.

Do not interpret a condition mean without checking the source-age distribution and story-level uncertainty.